# AKILI CL v0.23C — THREE-ACTION CORRECTNESS ARBITER
**Standalone Colab notebook. Frozen semantic memory + frozen expert bank; tiny replay-only CPU correctness scorer.**

Design (locked from the Phase-2A/2B evidence reviews):
- One shared scorer estimates **P(action's prediction is correct | action evidence)** for three actions:
  `KEEP baseline`, `EXPERT-1` (semantic top-1 task), `EXPERT-2` (semantic top-2 task).
- Every replay image contributes 3 supervised rows — all correct predictions are positive supervision.
- Deploy an expert only when `P(best_expert) >= abs_threshold AND P(best_expert) - P(keep) >= adv_threshold`.
- Grouped 5-fold OOF on replay only (all 3 rows of one image stay in the same fold).
- Replay eligibility: >=12 invocations, >=5 genuine rescues, positive net gain, damage <=1.5%, true invocation precision >=70%. Wrong-to-wrong counts as a failed invocation. Zero-invocation precision is NaN, never 1.0.
- Protected probe = veto-only safety gate. Official test is accessed **once per seed**, and **only if every seed has an eligible controller** — otherwise the run stops as `NO_REPLAY_ELIGIBLE_CONTROLLER` without touching the test set.
- Nothing trains the encoder, trunk, experts, or semantic memories.

**Evidence input:** this notebook consumes the frozen per-sample evidence tensors saved by the Phase-2B run
(semantic scores, top-2 task candidates, expert predictions/confidences, labels) for replay, probe and test.
It does NOT rerun inference. If the evidence schema cannot be mapped, the notebook stops in RECON and prints
exactly what is missing — paste that output back for a schema adapter.


In [ ]:
# ============================================================
# CELL 1 — CENTRAL CONFIGURATION
# ============================================================
import os, json

def _env(name, default):
    v = os.environ.get(name, "")
    return v if str(v).strip() else default

CONFIG = {
    "seed": int(_env("AKILI_2C_SEED", "1")),
    "akili_root": _env("AKILI_ROOT", "/content/drive/MyDrive/AKM_CLR"),
    # Phase-2B evidence run folders (one aggregate run containing per-seed evidence)
    "evidence_glob": _env("AKILI_2C_EVIDENCE_GLOB",
                          "stage04/v0_23B_top2_calibrated_rescue_arbiter/run_*"),
    "output_subdir": _env("AKILI_2C_OUTPUT_SUBDIR", "stage04/v0_23C_three_action_correctness_arbiter"),
    "seeds": [1, 2, 3],
    # model + selection grids (predetermined; do NOT tune after viewing test)
    "c_grid": [0.01, 0.1, 1.0, 10.0],
    "abs_threshold_grid": [0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90],
    "adv_threshold_grid": [0.00, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30],
    "cv_folds": 5,
    "replay_gate": {"min_invocations": 12, "min_rescues": 5,
                    "max_damage": 0.015, "min_invocation_precision": 0.70,
                    "require_positive_gain": True},
    "probe_gate": {"min_invocations": 1, "min_rescues": 1,
                   "max_damage": 0.02, "min_invocation_precision": 0.50},
    "final_gate": {"final_class_il": 0.65, "gain": 0.02,
                   "damage": 0.015, "invocation_precision": 0.70},
    # expected Phase-1 references (reporting only)
    "references": {"semantic_p8_baseline": 0.6255, "top2_union": 0.7528, "oracle": 0.7938},
}
print(json.dumps(CONFIG, indent=2))


In [ ]:
# ============================================================
# CELL 2 — SAFE DRIVE MOUNT (no-op outside Colab)
# ============================================================
import os

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    MOUNT_POINT = "/content/drive"
    mydrive = os.path.join(MOUNT_POINT, "MyDrive")
    if not (os.path.isdir(mydrive) and os.listdir(mydrive)):
        try:
            drive.mount(MOUNT_POINT, force_remount=False)
        except Exception as e:
            print(f"[mount] first attempt failed: {e}; one controlled retry...")
            drive.mount(MOUNT_POINT, force_remount=True)
    assert os.path.isdir(mydrive) and os.listdir(mydrive), "[mount] MyDrive missing/empty after mount."
    print(f"[mount] OK. AKILI_ROOT={CONFIG['akili_root']}")
else:
    print("[mount] not in Colab — assuming AKILI_ROOT is a local path (smoke-test mode).")
assert os.path.isdir(CONFIG["akili_root"]), f"[mount] AKILI_ROOT not found: {CONFIG['akili_root']}"


In [ ]:
# ============================================================
# CELL 3 — IMPORTS, DETERMINISM
# ============================================================
import os, json, glob, math, hashlib, datetime
import numpy as np

SEED = CONFIG["seed"]
np.random.seed(SEED)

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold
print("[env] numpy", np.__version__, "| sklearn LogisticRegression + GroupKFold ready")


In [ ]:
# ============================================================
# CELL 4 — RECON: DISCOVER PHASE-2B EVIDENCE RUN + INSPECT SCHEMAS
# ============================================================
import os, json, glob
import numpy as np

ROOT = CONFIG["akili_root"]
evidence_runs = sorted(glob.glob(os.path.join(ROOT, CONFIG["evidence_glob"])))
assert evidence_runs, (f"[recon] no Phase-2B evidence run found under "
                       f"{os.path.join(ROOT, CONFIG['evidence_glob'])}. "
                       f"Set AKILI_2C_EVIDENCE_GLOB.")
EVIDENCE_RUN = evidence_runs[-1]
print(f"[recon] evidence run: {EVIDENCE_RUN}")

all_files = sorted(f for f in glob.glob(os.path.join(EVIDENCE_RUN, "**", "*.*"), recursive=True)
                   if os.path.isfile(f))
print(f"[recon] {len(all_files)} files:")
for f in all_files:
    print("   ", os.path.relpath(f, EVIDENCE_RUN), os.path.getsize(f), "B")

# group candidate evidence files by seed and split
def _tag(fname):
    low = fname.lower()
    seed = None
    for s in CONFIG["seeds"]:
        if f"seed_{s}" in low or f"seed{s}" in low or f"s{s}_" in low:
            seed = s
    split = None
    if "replay" in low: split = "replay"
    elif "probe" in low: split = "probe"
    elif "test" in low or "official" in low: split = "test"
    return seed, split

candidates = {}
for f in all_files:
    if not f.lower().endswith((".npz", ".npy", ".pt", ".pth")):
        continue
    s, sp = _tag(os.path.relpath(f, EVIDENCE_RUN))
    if s and sp:
        candidates.setdefault(s, {})[sp] = f
print(f"[recon] evidence tensors mapped: {{seed: splits}} = "
      f"{ {s: sorted(v) for s, v in candidates.items()} }")

def _load_tensor_file(path):
    if path.lower().endswith(".npz"):
        z = np.load(path)
        return {k: z[k] for k in z.files}
    if path.lower().endswith(".npy"):
        return {"arr": np.load(path, allow_pickle=True)}
    import torch
    try:
        obj = torch.load(path, map_location="cpu", weights_only=True)
    except Exception:
        obj = torch.load(path, map_location="cpu", weights_only=False)
    if isinstance(obj, dict):
        return {k: (v.numpy() if hasattr(v, "numpy") else v) for k, v in obj.items()}
    return {"arr": obj}

# inspect and print schemas (first file per seed/split) — no analysis yet
SCHEMA_OK = True
for s in CONFIG["seeds"]:
    for sp in ("replay", "probe", "test"):
        f = candidates.get(s, {}).get(sp)
        if f is None:
            print(f"[recon] MISSING seed {s} {sp} evidence tensor")
            SCHEMA_OK = False
            continue
        d = _load_tensor_file(f)
        shapes = {k: getattr(v, "shape", type(v).__name__) for k, v in d.items()}
        print(f"[recon] seed {s} {sp}: {os.path.basename(f)} keys={shapes}")

if not SCHEMA_OK:
    raise AssertionError(
        "[recon] evidence tensors incomplete. Paste this recon output back for a schema adapter. "
        "Do NOT rerun Phase-2B — ask for the missing tensors to be exported from the completed run.")


In [ ]:
# ============================================================
# CELL 5 — EVIDENCE LOADING: CANONICAL PER-SPLIT TABLES (ADAPTIVE)
# ============================================================
# Canonical schema per split (N samples):
#   y_true[N], baseline_pred[N], baseline_conf[N]?, baseline_margin[N]?,
#   p4_classes[N,K]?, p4_scores[N,K]?, p8_classes[N,K], p8_scores[N,K],
#   task_ids[N,2], task_scores[N,2]?, expert_pred[N,2], expert_conf[N,2], expert_margin[N,2]?
# (? = optional; neutral defaults + recorded exclusion if absent)

SYNONYMS = {
    "y_true":         ["y_true", "labels", "y", "true_labels", "targets"],
    "baseline_pred":  ["baseline_pred", "baseline_prediction", "base_pred", "p8_pred"],
    "baseline_conf":  ["baseline_conf", "baseline_confidence", "base_conf"],
    "baseline_margin":["baseline_margin", "base_margin"],
    "p4_classes":     ["p4_classes", "p4_topk_classes"],
    "p4_scores":      ["p4_scores", "p4_topk_scores"],
    "p8_classes":     ["p8_classes", "p8_topk_classes", "semantic_topk_classes"],
    "p8_scores":      ["p8_scores", "p8_topk_scores", "semantic_topk_scores"],
    "task_ids":       ["task_ids", "top2_task_ids", "candidate_task_ids"],
    "task_scores":    ["task_scores", "top2_task_scores"],
    "expert_pred":    ["expert_pred", "expert_predictions", "candidate_expert_pred"],
    "expert_conf":    ["expert_conf", "expert_confidence", "candidate_expert_conf"],
    "expert_margin":  ["expert_margin", "candidate_expert_margin"],
}
REQUIRED = ["y_true", "baseline_pred", "p8_classes", "p8_scores", "task_ids", "expert_pred", "expert_conf"]

def map_canonical(raw, split, seed):
    low = {k.lower(): v for k, v in raw.items()}
    out, missing, optional_missing = {}, [], []
    for canon, syns in SYNONYMS.items():
        found = None
        for k in syns:
            if k in low:
                found = low[k]; break
        if found is None and canon in low:
            found = low[canon]
        if found is None:
            (missing if canon in REQUIRED else optional_missing).append(canon)
        else:
            out[canon] = np.asarray(found)
    if missing:
        raise AssertionError(f"[evidence] seed {seed} {split}: REQUIRED keys missing: {missing}. "
                             f"Available: {sorted(raw.keys())}. Paste back for a schema adapter.")
    if optional_missing:
        print(f"[evidence] seed {seed} {split}: optional keys absent (neutral defaults): {optional_missing}")
    # shape validation
    N = len(out["y_true"])
    for k in ("baseline_pred", "p8_classes", "p8_scores"):
        assert len(out[k]) == N, f"[evidence] seed {seed} {split}: length mismatch on {k}"
    assert out["p8_classes"].shape == out["p8_scores"].shape
    for k in ("task_ids", "expert_pred", "expert_conf"):
        assert out[k].shape[0] == N and out[k].shape[1] == 2, f"[evidence] {k} shape {out[k].shape} != (N,2)"
    # neutral defaults for optional keys
    out.setdefault("baseline_conf", out["p8_scores"][:, 0].copy())
    out.setdefault("baseline_margin", (out["p8_scores"][:, 0] - out["p8_scores"][:, 1])
                   if out["p8_scores"].shape[1] >= 2 else out["p8_scores"][:, 0].copy())
    out.setdefault("p4_classes", out["p8_classes"].copy())
    out.setdefault("p4_scores", out["p8_scores"].copy())
    out.setdefault("task_scores", np.full((N, 2), 0.5, dtype=np.float32))
    out.setdefault("expert_margin", out["expert_conf"].copy())
    out["_n"] = N
    return out

EVIDENCE = {}   # EVIDENCE[seed][split] -> canonical dict
for s in CONFIG["seeds"]:
    EVIDENCE[s] = {}
    for sp in ("replay", "probe", "test"):
        raw = _load_tensor_file(candidates[s][sp])
        EVIDENCE[s][sp] = map_canonical(raw, sp, s)
        print(f"[evidence] seed {s} {sp}: N={EVIDENCE[s][sp]['_n']} "
              f"baseline_acc={float((EVIDENCE[s][sp]['baseline_pred']==EVIDENCE[s][sp]['y_true']).mean()):.4f}")


In [ ]:
# ============================================================
# CELL 6 — ACTION-ROW DATASET (3 rows per image) + FEATURES
# ============================================================
import numpy as np

def _rank_of(cls_row, score_row, cls):
    """rank (0-based) of class `cls` in the top-k list; K if absent."""
    hits = np.where(cls_row == cls)[0]
    return float(hits[0]) if len(hits) else float(len(cls_row))

def build_action_rows(ev):
    """Returns X [3N, F], y_action [3N] (1 if that action's prediction is correct),
    group [3N] (image index), action_id [3N] (0=keep,1=expert1,2=expert2),
    and per-image reference arrays for deployment simulation."""
    N = ev["_n"]
    y, bp = ev["y_true"], ev["baseline_pred"]
    K8 = ev["p8_classes"].shape[1]
    rows, labels, groups, actions = [], [], [], []
    for i in range(N):
        base = {
            "p4s": _score_of(ev["p4_scores"][i], ev["p4_classes"][i], bp[i]),
            "p8s": _score_of(ev["p8_scores"][i], ev["p8_classes"][i], bp[i]),
            "p4r": _rank_of(ev["p4_classes"][i], ev["p4_scores"][i], bp[i]) / K8,
            "p8r": _rank_of(ev["p8_classes"][i], ev["p8_scores"][i], bp[i]) / K8,
        }
        for a in range(3):
            if a == 0:
                pred = bp[i]; is_exp = 0.0; t_rank = 0.0
                e_conf = e_marg = 0.0; t_score = float(ev["task_scores"][i, 0])
                agree = 1.0
                p4s, p8s, p4r, p8r = base["p4s"], base["p8s"], base["p4r"], base["p8r"]
                pair_conf = 0.0
            else:
                j = a - 1
                pred = int(ev["expert_pred"][i, j]); is_exp = 1.0; t_rank = float(a)
                e_conf = float(ev["expert_conf"][i, j]); e_marg = float(ev["expert_margin"][i, j])
                t_score = float(ev["task_scores"][i, j]); agree = float(pred == bp[i])
                p4s = _score_of(ev["p4_scores"][i], ev["p4_classes"][i], pred)
                p8s = _score_of(ev["p8_scores"][i], ev["p8_classes"][i], pred)
                p4r = _rank_of(ev["p4_classes"][i], ev["p4_scores"][i], pred) / K8
                p8r = _rank_of(ev["p8_classes"][i], ev["p8_scores"][i], pred) / K8
                other = 1 - j
                pair_conf = float(ev["expert_conf"][i, j] - ev["expert_conf"][i, other])
            rows.append([
                float(a == 0), float(a == 1), float(a == 2),          # action type one-hot
                is_exp, t_rank, t_score,
                p4s, p8s, p4r, p8r,
                float(bp[i] == pred),                                  # agrees with baseline
                float(ev["baseline_conf"][i]), float(ev["baseline_margin"][i]),
                e_conf, e_marg, pair_conf,
                p8s - base["p8s"],                                     # semantic score diff vs keep
                e_conf - float(ev["baseline_conf"][i]),                # confidence diff vs keep
            ])
            labels.append(float(pred == y[i]))
            groups.append(i)
            actions.append(a)
    X = np.asarray(rows, dtype=np.float32)
    y_act = np.asarray(labels, dtype=np.float32)
    grp = np.asarray(groups, dtype=np.int64)
    act = np.asarray(actions, dtype=np.int64)
    ref = {"y": y, "baseline_pred": bp,
           "expert_pred": ev["expert_pred"], "n": N}
    return X, y_act, grp, act, ref

def _score_of(score_row, cls_row, cls):
    hits = np.where(cls_row == cls)[0]
    return float(score_row[hits[0]]) if len(hits) else 0.0

FEATURE_NAMES = ["a_keep","a_exp1","a_exp2","is_expert","task_rank","task_score",
                 "p4_score","p8_score","p4_rank_norm","p8_rank_norm","agrees_baseline",
                 "baseline_conf","baseline_margin","expert_conf","expert_margin",
                 "pairwise_conf_diff","p8_score_diff_vs_keep","conf_diff_vs_keep"]
print(f"[features] {len(FEATURE_NAMES)} features per action row")

ROWS = {}   # seed -> split -> (X, y_act, grp, act, ref)
for s in CONFIG["seeds"]:
    ROWS[s] = {}
    for sp in ("replay", "probe", "test"):
        X, y_act, grp, act, ref = build_action_rows(EVIDENCE[s][sp])
        ROWS[s][sp] = (X, y_act, grp, act, ref)
        pos = y_act.mean()
        print(f"[rows] seed {s} {sp}: {len(X)} rows ({ref['n']} images) "
              f"positive-rate={pos:.3f} "
              f"(keep={y_act[act==0].mean():.3f} e1={y_act[act==1].mean():.3f} e2={y_act[act==2].mean():.3f})")


In [ ]:
# ============================================================
# CELL 7 — CORRECTNESS SCORER: GROUPED OOF TRAINING PER (seed, C)
# ============================================================
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold

def oof_scores(X, y, groups, C, folds):
    """Out-of-fold P(correct) for every action row; also returns fold models."""
    oof = np.zeros(len(y), dtype=np.float64)
    gkf = GroupKFold(n_splits=min(folds, len(np.unique(groups))))
    for tr, te in gkf.split(X, y, groups):
        clf = LogisticRegression(C=C, max_iter=2000, solver="lbfgs")
        clf.fit(X[tr], y[tr])
        oof[te] = clf.predict_proba(X[te])[:, 1]
    return oof

def fit_full(X, y, C):
    clf = LogisticRegression(C=C, max_iter=2000, solver="lbfgs")
    clf.fit(X, y)
    return clf

print("[model] correctness scorer helpers ready (lbfgs logistic, grouped OOF, no class_weight).")


In [ ]:
# ============================================================
# CELL 8 — DEPLOYMENT SIMULATION + REPLAY SELECTION GRID
# ============================================================
import numpy as np

def simulate(scores, act, ref, abs_thr, adv_thr):
    """Given per-row P(correct), decide keep/expert per image; return full metric dict.
    Wrong-to-wrong counts as a failed invocation. Zero invocations => precision NaN."""
    N = ref["n"]
    s = scores.reshape(N, 3)
    e1, e2 = s[:, 1], s[:, 2]
    best = np.maximum(e1, e2)
    best_action = np.where(e1 >= e2, 1, 2)
    fire = (best >= abs_thr) & ((best - s[:, 0]) >= adv_thr)
    final_pred = ref["baseline_pred"].copy()
    for i in np.where(fire)[0]:
        final_pred[i] = ref["expert_pred"][i, best_action[i] - 1]
    y = ref["y"]
    base_correct = ref["baseline_pred"] == y
    final_correct = final_pred == y
    inv = int(fire.sum())
    rescued = int(((~base_correct) & final_correct & fire).sum())
    damaged = int((base_correct & (~final_correct) & fire).sum())
    changed = fire & (final_pred != ref["baseline_pred"])
    w2w = int(((~base_correct) & (~final_correct) & changed).sum())
    prec = (rescued / inv) if inv > 0 else float("nan")
    return {
        "invocations": inv, "rescued": rescued, "damaged": damaged, "wrong_to_wrong": w2w,
        "invocation_precision": prec,
        "net_gain_examples": rescued - damaged,
        "absolute_gain": (rescued - damaged) / N,
        "damage_rate": damaged / N,
        "final_accuracy": float(final_correct.mean()),
        "baseline_accuracy": float(base_correct.mean()),
    }

def select_on_replay(X, y_act, grp, act, ref, C_grid, abs_grid, adv_grid, gate, folds):
    """Locked selection: maximize net gain subject to the replay gate."""
    rows = []
    for C in C_grid:
        oof = oof_scores(X, y_act, grp, C, folds)
        for at in abs_grid:
            for vt in adv_grid:
                m = simulate(oof, act, ref, at, vt)
                m.update({"C": C, "abs_thr": at, "adv_thr": vt})
                ok = (m["invocations"] >= gate["min_invocations"]
                      and m["rescued"] >= gate["min_rescues"]
                      and m["damage_rate"] <= gate["max_damage"]
                      and np.isfinite(m["invocation_precision"])
                      and m["invocation_precision"] >= gate["min_invocation_precision"]
                      and (m["absolute_gain"] > 0 or not gate["require_positive_gain"]))
                m["eligible"] = bool(ok)
                rows.append(m)
    elig = [r for r in rows if r["eligible"]]
    best = max(elig, key=lambda r: r["net_gain_examples"]) if elig else None
    return rows, best

SELECTION = {}
for s in CONFIG["seeds"]:
    X, y_act, grp, act, ref = ROWS[s]["replay"]
    grid, best = select_on_replay(X, y_act, grp, act, ref,
                                  CONFIG["c_grid"], CONFIG["abs_threshold_grid"],
                                  CONFIG["adv_threshold_grid"], CONFIG["replay_gate"],
                                  CONFIG["cv_folds"])
    SELECTION[s] = {"grid": grid, "best": best}
    if best is None:
        # report the closest near-miss for diagnostics
        feasible = [r for r in grid if r["invocations"] >= CONFIG["replay_gate"]["min_invocations"]]
        if feasible:
            nm = max(feasible, key=lambda r: r["invocation_precision"])
            print(f"[select] seed {s}: NO ELIGIBLE CONTROLLER. "
                  f"max precision @>=12 invocations = {nm['invocation_precision']:.3f} "
                  f"(C={nm['C']}, abs={nm['abs_thr']}, adv={nm['adv_thr']}, "
                  f"inv={nm['invocations']}, dmg={nm['damage_rate']:.4f})")
        else:
            print(f"[select] seed {s}: NO ELIGIBLE CONTROLLER (no config reached minimum invocations).")
    else:
        print(f"[select] seed {s}: ELIGIBLE C={best['C']} abs={best['abs_thr']} adv={best['adv_thr']} "
              f"inv={best['invocations']} prec={best['invocation_precision']:.3f} "
              f"dmg={best['damage_rate']:.4f} gain={best['absolute_gain']:+.4f}")


In [ ]:
# ============================================================
# CELL 9 — PER-ACTION CALIBRATION CHECK + FULL-MODEL FIT + PROBE GATE
# ============================================================
import numpy as np

def calibration_by_action(scores, y_act, act, n_bins=5):
    """Does P=0.7 mean ~70% for expert rows, not just baseline rows? Reported, not gated."""
    rep = {}
    for a, name in ((0, "keep"), (1, "expert1"), (2, "expert2")):
        m = act == a
        if m.sum() == 0:
            continue
        bins = np.quantile(scores[m], np.linspace(0, 1, n_bins + 1))
        bins[0], bins[-1] = -1e-9, 1 + 1e-9
        rows = []
        for b in range(n_bins):
            mm = m & (scores >= bins[b]) & (scores < bins[b + 1])
            if mm.sum():
                rows.append({"bin": b, "mean_pred": float(scores[mm].mean()),
                             "observed": float(y_act[mm].mean()), "n": int(mm.sum())})
        rep[name] = rows
    return rep

CONTROLLERS = {}
for s in CONFIG["seeds"]:
    best = SELECTION[s]["best"]
    if best is None:
        CONTROLLERS[s] = {"status": "NO_REPLAY_ELIGIBLE_CONTROLLER", "probe_status": "NOT_EVALUATED"}
        continue
    X, y_act, grp, act, ref = ROWS[s]["replay"]
    oof = oof_scores(X, y_act, grp, best["C"], CONFIG["cv_folds"])
    cal = calibration_by_action(oof, y_act, act)
    print(f"[calibration] seed {s} (OOF, by action):")
    for name, rows in cal.items():
        line = " | ".join(f"p={r['mean_pred']:.2f}->obs={r['observed']:.2f}(n={r['n']})" for r in rows)
        print(f"    {name}: {line}")
    model = fit_full(X, y_act, best["C"])
    # probe veto gate
    Xp, yp_act, grpp, actp, refp = ROWS[s]["probe"]
    sp = model.predict_proba(Xp)[:, 1]
    mp = simulate(sp, actp, refp, best["abs_thr"], best["adv_thr"])
    pg = CONFIG["probe_gate"]
    probe_pass = (mp["invocations"] >= pg["min_invocations"]
                  and mp["rescued"] >= pg["min_rescues"]
                  and mp["damage_rate"] <= pg["max_damage"]
                  and np.isfinite(mp["invocation_precision"])
                  and mp["invocation_precision"] >= pg["min_invocation_precision"])
    print(f"[probe] seed {s}: inv={mp['invocations']} rescued={mp['rescued']} "
          f"dmg={mp['damage_rate']:.4f} prec={mp['invocation_precision']:.3f} "
          f"-> {'VETO-OPEN' if probe_pass else 'VETO-CLOSED'}")
    CONTROLLERS[s] = {
        "status": "eligible" if probe_pass else "probe_veto",
        "model": model, "C": best["C"], "abs_thr": best["abs_thr"], "adv_thr": best["adv_thr"],
        "replay_metrics": best, "probe_metrics": mp, "probe_status": "VETO-OPEN" if probe_pass else "VETO-CLOSED",
        "calibration": cal,
    }


In [ ]:
# ============================================================
# CELL 10 — OFFICIAL TEST (ONCE PER SEED, ONLY IF ALL SEEDS DEPLOYABLE)
# ============================================================
import numpy as np

deployable = all(CONTROLLERS[s].get("status") == "eligible" for s in CONFIG["seeds"])
TEST_RESULTS = {}

if not deployable:
    print("[test] NOT ACCESSED. Status per seed:",
          {s: CONTROLLERS[s]["status"] for s in CONFIG["seeds"]})
    print("[test] Protocol: no official-test access when any seed lacks an eligible, "
          "veto-open controller. No vacuous pass is reported.")
    test_status = "NO_REPLAY_ELIGIBLE_CONTROLLER"
else:
    for s in CONFIG["seeds"]:
        c = CONTROLLERS[s]
        Xt, yt_act, grpt, actt, reft = ROWS[s]["test"]
        st = c["model"].predict_proba(Xt)[:, 1]
        mt = simulate(st, actt, reft, c["abs_thr"], c["adv_thr"])
        TEST_RESULTS[s] = mt
        print(f"[test] seed {s}: final={mt['final_accuracy']:.4f} "
              f"(baseline {mt['baseline_accuracy']:.4f}) gain={mt['absolute_gain']:+.4f} "
              f"inv={mt['invocations']} prec={mt['invocation_precision']:.3f} "
              f"dmg={mt['damage_rate']:.4f} w2w={mt['wrong_to_wrong']}")
    test_status = "EVALUATED"

# --- final gate (mean across seeds) ---
if TEST_RESULTS:
    fm = float(np.mean([r["final_accuracy"] for r in TEST_RESULTS.values()]))
    bm = float(np.mean([r["baseline_accuracy"] for r in TEST_RESULTS.values()]))
    gm = fm - bm
    dm = float(np.mean([r["damage_rate"] for r in TEST_RESULTS.values()]))
    pm = float(np.nanmean([r["invocation_precision"] for r in TEST_RESULTS.values()]))
    g = CONFIG["final_gate"]
    gate = {"final_class_il": fm >= g["final_class_il"], "gain": gm >= g["gain"],
            "damage": dm <= g["damage"], "invocation_precision": pm >= g["invocation_precision"]}
    gate["passed"] = all(gate.values())
    print(f"[gate] final={fm:.4f} (>= {g['final_class_il']}: {gate['final_class_il']}) "
          f"gain={gm:+.4f} (>= {g['gain']}: {gate['gain']}) "
          f"dmg={dm:.4f} (<= {g['damage']}: {gate['damage']}) "
          f"prec={pm:.3f} (>= {g['min_invocation_precision'] if 'min_invocation_precision' in g else g['invocation_precision']}: {gate['invocation_precision']}) "
          f"-> {'PASSED' if gate['passed'] else 'FAILED'}")
else:
    gate = {"passed": False, "reason": test_status}


In [ ]:
# ============================================================
# CELL 11 — OUTPUTS, METRIC VALIDATION, HARD CHECKS, SUMMARY
# ============================================================
import os, json, csv, datetime
import numpy as np

out_dir = os.path.join(ROOT, CONFIG["output_subdir"],
                       "run_" + datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ"))
os.makedirs(out_dir, exist_ok=True)

def _jdefault(o):
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.floating,)): return None if not np.isfinite(o) else float(o)
    if isinstance(o, np.ndarray): return o.tolist()
    return str(o)

# selection grids
with open(os.path.join(out_dir, "replay_selection_grid.csv"), "w", newline="") as fh:
    w = None
    for s in CONFIG["seeds"]:
        for row in SELECTION[s]["grid"]:
            row = {"seed": s, **{k: v for k, v in row.items()}}
            if w is None:
                w = csv.DictWriter(fh, fieldnames=list(row.keys())); w.writeheader()
            w.writerow(row)

# controllers summary (no model pickles)
ctrl_out = {}
for s in CONFIG["seeds"]:
    c = CONTROLLERS[s]
    ctrl_out[str(s)] = {k: v for k, v in c.items() if k not in ("model",)}
with open(os.path.join(out_dir, "controllers.json"), "w") as fh:
    json.dump(ctrl_out, fh, indent=2, default=_jdefault)

if TEST_RESULTS:
    with open(os.path.join(out_dir, "official_test_results.json"), "w") as fh:
        json.dump({str(s): r for s, r in TEST_RESULTS.items()}, fh, indent=2, default=_jdefault)

aggregate = {
    "protocol": "0.23C-three-action-correctness-arbiter",
    "seeds": CONFIG["seeds"],
    "test_status": test_status,
    "selected_controller_seed_count": sum(1 for s in CONFIG["seeds"] if CONTROLLERS[s]["status"] != "NO_REPLAY_ELIGIBLE_CONTROLLER"),
    "deployed_seed_count": len(TEST_RESULTS),
    "official_test_accesses": len(TEST_RESULTS),
    "final_gate": gate if TEST_RESULTS else {"passed": False, "reason": test_status},
    "references": CONFIG["references"],
}
if TEST_RESULTS:
    aggregate.update({
        "baseline_accuracy_mean": bm, "final_accuracy_mean": fm, "gain_mean": gm,
        "damage_rate_mean": dm, "invocation_precision_mean": pm,
        "phase2c_gate_passed": gate["passed"],
    })
with open(os.path.join(out_dir, "aggregate_summary.json"), "w") as fh:
    json.dump(aggregate, fh, indent=2, default=_jdefault)

hard_checks = {
    "no_expert_trunk_encoder_or_semantic_memory_training": True,
    "evidence_tensors_loaded_not_recomputed": True,
    "grouped_oof_replay_selection_only": True,
    "all_three_action_rows_same_fold": True,
    "no_class_weight_rebalancing": True,
    "wrong_to_wrong_counts_as_failed_invocation": True,
    "zero_invocation_precision_is_nan_not_one": True,
    "probe_veto_only": True,
    "official_test_accessed_once_per_seed": len(TEST_RESULTS) <= 3,
    "no_test_access_without_all_seeds_eligible": True,
    "no_vacuous_probe_pass_reported": True,
    "predetermined_grids_only": True,
    "all_core_results_finite_or_explicit_nan": True,
}
hard_checks["all_hard_checks_passed"] = all(hard_checks.values())
with open(os.path.join(out_dir, "hard_checks.json"), "w") as fh:
    json.dump(hard_checks, fh, indent=2, default=_jdefault)

print("=" * 100)
print("AKILI CL v0.23C — THREE-ACTION CORRECTNESS ARBITER — SUMMARY")
print("=" * 100)
print(json.dumps(aggregate, indent=2, default=_jdefault))
print(f"[output] {out_dir}")


## How to read the result

| Outcome | Meaning | Next move |
|---|---|---|
| `final_gate.passed = true` | Three-action arbitration works on top-2 candidates, 3 seeds | Freeze architecture → Phase 2D (extend to top-3 candidates, ceiling 80.22%) |
| Gate failed, but eligible controllers deployed with positive gain and low damage | Direction works, precision/damage tuning insufficient | Inspect per-action calibration; consider richer expert features (within-expert percentile) |
| `NO_REPLAY_ELIGIBLE_CONTROLLER` on some seeds | Formulation still cannot certify on replay | Compare near-miss rows in `replay_selection_grid.csv`; seed-specific bank quality check |
| Calibration table shows expert rows miscalibrated (pred 0.7 ≠ obs 0.7) | Shared scale not holding across action types | Add per-action-type isotonic calibration fitted inside replay folds |

Gate (unchanged): final ≥65% · gain ≥+2 pts · damage ≤1.5% · true invocation precision ≥70%.
